# GCN Example — Node Classification on the Karate Club Graph

A minimal **Graph Convolutional Network (GCN)** (Kipf & Welling, 2017) built from
scratch with TensorFlow / Keras — no graph libraries required.

We use **Zachary's Karate Club**: a famous 34-node social network that split into two
factions. The task is *semi-supervised node classification* — we only know the label of
**2 nodes** (the instructor and the club president) and let the GCN propagate that
information across the graph edges to classify everyone else.

A GCN layer is just:

$$H' = \sigma\big(\hat{A}\,H\,W\big)$$

where $\hat{A}$ is the normalized adjacency matrix (each node mixes in its neighbours'
features), $H$ are the node features, and $W$ is a learned weight matrix.

> Tested with `tensorflow==2.1` (matches `requirements.txt`). Runs instantly on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)

## 1. Build the Karate Club graph

We hard-code the 78 edges so the notebook has no dependency on `networkx`.
Node 0 is the instructor ("Mr. Hi") and node 33 is the club president ("Officer").

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]

N = 34  # number of nodes

# Symmetric adjacency matrix
A = np.zeros((N, N), dtype='float32')
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

print('Nodes:', N, '| Edges:', len(edges))

## 2. Normalize the adjacency matrix

GCNs use $\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2}$ where
$\tilde{A} = A + I$ (add self-loops) and $\tilde{D}$ is its degree matrix. This keeps
feature scales stable as they propagate across the graph.

In [ ]:
A_tilde = A + np.eye(N, dtype='float32')        # add self-loops
D = np.sum(A_tilde, axis=1)                      # degree of each node
D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
A_hat = (D_inv_sqrt @ A_tilde @ D_inv_sqrt).astype('float32')
A_hat = tf.constant(A_hat)

print('Normalized adjacency shape:', A_hat.shape)

## 3. Features and labels

We have no real node features, so we use a one-hot identity matrix (a common trick — the
GCN then learns an embedding per node). Only nodes **0** and **33** are labelled.

In [ ]:
X = tf.eye(N)  # identity features, shape (34, 34)

# Ground-truth factions (used only for final evaluation / plotting)
y_true = np.array([
    0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,
    1,1,1,1,1,1,1,1,1,1
])

# Semi-supervised: we only 'know' two labels
labeled_nodes = [0, 33]
y_labeled = tf.constant([y_true[0], y_true[33]])

print('Labeled nodes:', labeled_nodes, '-> labels', y_labeled.numpy())

## 4. Define a GCN layer

A custom Keras layer that computes $\hat{A}\,(X W)$. The normalized adjacency `A_hat`
is captured at construction time.

In [ ]:
class GCNLayer(layers.Layer):
    def __init__(self, units, A_hat, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.A_hat = A_hat
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='w',
        )

    def call(self, x):
        x = tf.matmul(x, self.w)         # X W
        x = tf.matmul(self.A_hat, x)     # A_hat (X W)  -> mix neighbours
        return self.activation(x) if self.activation else x

## 5. Build the 2-layer GCN

Layer 1 maps the 34-dim features to a 16-dim hidden representation (ReLU); layer 2 maps
to 2 output classes.

In [ ]:
class GCN(tf.keras.Model):
    def __init__(self, A_hat, hidden=16, classes=2):
        super().__init__()
        self.gc1 = GCNLayer(hidden, A_hat, activation='relu')
        self.gc2 = GCNLayer(classes, A_hat, activation=None)

    def call(self, x):
        x = self.gc1(x)
        self.embeddings = x          # keep hidden embeddings for plotting
        return self.gc2(x)

model = GCN(A_hat)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(0.05)

## 6. Train

The loss is computed **only on the 2 labelled nodes**, but because the GCN mixes
neighbour information at every layer, gradients flow to the whole graph.

In [ ]:
EPOCHS = 200

for epoch in range(1, EPOCHS + 1):
    with tf.GradientTape() as tape:
        logits = model(X)
        logits_labeled = tf.gather(logits, labeled_nodes)
        loss = loss_fn(y_labeled, logits_labeled)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch % 20 == 0 or epoch == 1:
        preds = tf.argmax(model(X), axis=1).numpy()
        acc = (preds == y_true).mean()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={float(loss):.4f}  full-graph acc={acc:.3f}')

## 7. Results

From just two labelled nodes the GCN classifies the rest of the club. Let's check
accuracy against the true factions.

In [ ]:
preds = tf.argmax(model(X), axis=1).numpy()
acc = (preds == y_true).mean()
print(f'Final accuracy on all {N} nodes: {acc:.1%}')
print('Predicted factions:', preds)
print('True factions:     ', y_true)

## 8. Visualize the learned embeddings

The 16-dim hidden embeddings, projected to 2D with PCA, should separate the two
factions cleanly. The two labelled nodes are marked with a black border.

In [ ]:
_ = model(X)  # populate model.embeddings
emb = model.embeddings.numpy()

# Simple 2D PCA (no sklearn dependency)
emb_centered = emb - emb.mean(axis=0)
u, s, vt = np.linalg.svd(emb_centered, full_matrices=False)
coords = emb_centered @ vt[:2].T

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=preds, cmap='coolwarm', s=200, edgecolors='gray')
for n in labeled_nodes:
    plt.scatter(coords[n, 0], coords[n, 1], s=320, facecolors='none', edgecolors='black', linewidths=2.5)
for i, (x_, y_) in enumerate(coords):
    plt.text(x_, y_, str(i), ha='center', va='center', fontsize=8)
plt.title('GCN node embeddings (PCA) — colored by predicted faction')
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.show()

### Things to try

- Change `hidden` units or `EPOCHS` and see how the separation changes.
- Label different/more nodes and observe the effect on accuracy.
- Swap the identity features for real node attributes if you have them.
- Add a third GCN layer — does deeper help or hurt on such a small graph?